# Seasonal extreme rainfall spatial distribution

This notebook keeps the plotting code editable in-place. The expensive seasonal diagnostics are cached in NetCDF by default, including seasonal mean rainfall, seasonal 95th and 99th percentile rainfall, and seasonal 850 hPa winds.

In [ ]:
from pathlib import Path
import os
import importlib.util
import sys
import warnings

os.environ.setdefault('MPLCONFIGDIR', '/tmp/m301257_matplotlib')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

warnings.filterwarnings('ignore', message='pyproj unable to set PROJ database path')

BASE_DIR = Path('/work/mh1498/m301257')
CODE_DIR = BASE_DIR / 'code_extreme_event'
SCRIPT_DIR = CODE_DIR / 'scripts'
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
PROJECT_DATA_DIR = CODE_DIR / 'data'
PROJECT_PRESSURE_DIR = PROJECT_DATA_DIR / 'wind_850hpa_lat30'
FIG_DIR = CODE_DIR / 'figures'
SCRIPT_FIG_DIR = FIG_DIR / 'plot_extreme_rainfall_spatial_distribution'
FIELD_DIR = PROJECT_DATA_DIR / 'seasonal_fields'
CNTL_FIELDS_PATH = FIELD_DIR / 'seasonal_rainfall_wind_cntl_full_domain_fields.nc'
P4K_FIELDS_PATH = FIELD_DIR / 'seasonal_rainfall_wind_p4k_full_domain_fields.nc'
DIFF_FIELDS_PATH = FIELD_DIR / 'seasonal_rainfall_wind_p4k_minus_cntl_fields.nc'
CNTL_P99_FIELDS_PATH = FIELD_DIR / 'seasonal_rainfall_p99_cntl_full_domain_fields.nc'
P4K_P99_FIELDS_PATH = FIELD_DIR / 'seasonal_rainfall_p99_p4k_full_domain_fields.nc'

CNTL_INPUTS = {
    'pr_path': BASE_DIR / 'processed_data_lat_30/2d_layers/pr_cntl/pr_2deg_interp.nc',
    'u_path': PROJECT_PRESSURE_DIR / 'ua_850hpa_cntl_lat30.nc',
    'v_path': PROJECT_PRESSURE_DIR / 'va_850hpa_cntl_lat30.nc',
}
P4K_INPUTS = {
    'pr_path': BASE_DIR / 'processed_data_lat_30/2d_layers/pr_p4k/pr_2deg_interp.nc',
    'u_path': PROJECT_PRESSURE_DIR / 'ua_850hpa_p4k_lat30.nc',
    'v_path': PROJECT_PRESSURE_DIR / 'va_850hpa_p4k_lat30.nc',
}

for path in (BASE_DIR, CODE_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from plot_extreme_rainfall_spatial_distribution import parse_args, run_from_config

EASYXP_PATH = BASE_DIR / 'wave_tools' / 'easyxp.py'
_easyxp_spec = importlib.util.spec_from_file_location('easyxp_local', EASYXP_PATH)
if _easyxp_spec is None or _easyxp_spec.loader is None:
    raise ImportError(f'Cannot load quiver legend helper from {EASYXP_PATH}')
_easyxp_module = importlib.util.module_from_spec(_easyxp_spec)
_easyxp_spec.loader.exec_module(_easyxp_module)
simple_quiver_legend = _easyxp_module.simple_quiver_legend

print(f'Notebook working directory: {CODE_DIR}')

## Data / Cache

Run `./run_project_extreme_rainfall_pipeline.sh` on the compute node to regenerate project-local 30S-30N model-level data, explicit 850 hPa wind files, seasonal rainfall percentile fields, figures, and P4K-CNTL differences. Leave `USE_CACHED_FIELDS=True` when only changing figure style. Set it to `False` after changing input data paths, date range, pressure level, or precipitation units. If the cache does not contain `rain_p99`, the notebook will recompute it automatically.

In [ ]:
USE_CACHED_FIELDS = True
REQUIRED_DATA_VARS = {'rain_mean', 'rain_p95', 'rain_p99', 'u850', 'v850'}

def cache_is_complete(path):
    if not path.exists():
        return False
    with xr.open_dataset(path) as cached:
        has_vars = REQUIRED_DATA_VARS.issubset(set(cached.data_vars))
        has_seasons = 'season' in cached.coords and set(cached['season'].values.astype(str)) == {'DJF', 'MAM', 'JJA', 'SON'}
        p95_ok = has_vars and 'season' in cached['rain_p95'].dims
        p99_ok = has_vars and 'season' in cached['rain_p99'].dims
        wide_rain = has_vars and cached.sizes.get('lon', 0) >= 100 and cached.sizes.get('lat', 0) >= 30
        wide_wind = has_vars and cached.sizes.get('wind_lon', 0) >= 100 and cached.sizes.get('wind_lat', 0) >= 30
    return has_vars and has_seasons and p95_ok and p99_ok and wide_rain and wide_wind

def scenario_args(inputs, fields_name, output_name):
    return parse_args([
        '--pr-path', str(inputs['pr_path']),
        '--u-path', str(inputs['u_path']),
        '--v-path', str(inputs['v_path']),
        '--scheduler', 'threads',
        '--workers', 'auto',
        '--threads-per-worker', '4',
        '--wind-step', '2',
        '--dpi', '300',
        '--fields-name', fields_name,
        '--output-name', output_name,
    ])

def load_or_compute_scenario(label, inputs, fields_path):
    if USE_CACHED_FIELDS and cache_is_complete(fields_path):
        print(f'Loading cached {label} fields: {fields_path}')
        return xr.open_dataset(fields_path)

    print(f'Recomputing {label} seasonal rainfall, p95/p99, and wind fields over full input domain...')
    args = scenario_args(
        inputs,
        fields_path.name,
        f'figure1_style_seasonal_rainfall_wind_icon_{label.lower()}.png',
    )
    _, computed_path = run_from_config(args)
    return xr.open_dataset(computed_path)

def scenario_difference(warm, base):
    out = xr.Dataset()
    for var in sorted(REQUIRED_DATA_VARS):
        warm_var, base_var = xr.align(warm[var], base[var], join='inner')
        out[var] = (warm_var - base_var).astype('float32')
        out[var].attrs['units'] = warm[var].attrs.get('units', '')
        out[var].attrs['description'] = f'P4K minus CNTL {var}'
    out.attrs['description'] = 'Seasonal P4K minus CNTL rainfall and 850 hPa wind differences'
    out.attrs['difference'] = 'P4K - CNTL'
    return out

ds = load_or_compute_scenario('CNTL', CNTL_INPUTS, CNTL_FIELDS_PATH)
ds_p4k = load_or_compute_scenario('P4K', P4K_INPUTS, P4K_FIELDS_PATH)
ds_diff = scenario_difference(ds_p4k, ds)
ds_diff.to_netcdf(DIFF_FIELDS_PATH)
ds[['rain_p99']].load().to_netcdf(CNTL_P99_FIELDS_PATH)
ds_p4k[['rain_p99']].load().to_netcdf(P4K_P99_FIELDS_PATH)
print(f'Saved CNTL P99 fields: {CNTL_P99_FIELDS_PATH}')
print(f'Saved P4K P99 fields: {P4K_P99_FIELDS_PATH}')
print(f'Saved P4K-CNTL fields: {DIFF_FIELDS_PATH}')
print(ds)
print(ds_p4k)
print(ds_diff)
ds

## Plot Controls

The values below are the main knobs for AGU-style figure tuning: figure size, fonts, line weights, color levels, quiver density, quiver legend position, and output filenames.

In [ ]:
SEASONS = ('DJF', 'MAM', 'JJA', 'SON')
PANEL_LABELS = ('(a)', '(b)', '(c)', '(d)', '(e)', '(f)')
COMPARE_LABELS = ('(a)', '(b)', '(c)', '(d)')
PERCENTILE_MAP_LABELS = ('(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)', '(h)')
WARMING_CHANGE_LABELS = tuple(f'({chr(97 + idx)})' for idx in range(12))

# Saved NetCDF fields use the full input data domain. Change only this tuple to
# freely adjust the map view without recomputing the seasonal statistics.
PLOT_DOMAIN = (94.0, 136.0, -14.0, 25.0)
DOMAIN = PLOT_DOMAIN

MEAN_LEVELS = np.array([0.3, 0.75, 1.5, 3.0, 6.0, 12.0, 24.0])
P95_LEVELS = np.array([0.3, 0.75, 1.5, 3.0, 6.0, 12.0, 24.0, 48.0, 96.0])
P99_LEVELS = np.array([0.3, 0.75, 1.5, 3.0, 6.0, 12.0, 24.0, 48.0, 96.0, 144.0])
RATIO_LEVELS = np.array([1.0, 1.1, 1.2, 1.35, 1.5, 1.75, 2.0, 2.5, 3.0])
DIFF_LEVELS = np.array([0.0, 2.0, 4.0, 6.0, 8.0, 12.0, 16.0, 24.0, 32.0, 48.0])
MEAN_CHANGE_LEVELS = np.array([-6.0, -4.0, -3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0, 4.0, 6.0])
EXTREME_CHANGE_LEVELS = np.array([-48.0, -32.0, -24.0, -16.0, -12.0, -8.0, -4.0, -2.0, 0.0, 2.0, 4.0, 8.0, 12.0, 16.0, 24.0, 32.0, 48.0])

EXTREME_COMPARE_MODE = 'ratio'  # 'ratio' for P99/P95, or 'difference' for P99-P95.

PLOT = {
    'figure_width_in': 7.15,
    'figure_height_in': 10.7,
    'compare_width_in': 7.15,
    'compare_height_in': 6.6,
    'percentile_maps_width_in': 10.2,
    'percentile_maps_height_in': 7.0,
    'warming_change_width_in': 10.2,
    'warming_change_height_in': 10.8,
    'font_family': 'DejaVu Sans',
    'base_fontsize': 7.5,
    'title_fontsize': 9.0,
    'panel_label_fontsize': 10.0,
    'coastline_lw': 0.65,
    'border_lw': 0.35,
    'grid_lw': 0.35,
    'map_spine_lw': 1.5,
    'box_lw': 1.0,
    'wind_step': 2,
    'wind_scale': 80,
    'wind_width': 0.0020,
    'wind_key': 5.0,
    'wind_legend_location': 'lower right',
    'change_wind_key': 2.0,
    'change_wind_scale': 45,
    'lon_tick_step': 10,
    'lat_tick_step': 10,
    'cmap_name': 'cmocean_rain',
    'compare_cmap_name': 'cmocean_matter',
    'change_cmap_name': 'cmocean_balance',
    'save_dpi': 600,
    'save_main_png': FIG_DIR / 'figure1_style_AGU_editable_notebook.png',
    'save_main_pdf': FIG_DIR / 'figure1_style_AGU_editable_notebook.pdf',
    'save_compare_png': FIG_DIR / 'figure_extreme_p99_p95_compare_2x2.png',
    'save_compare_pdf': FIG_DIR / 'figure_extreme_p99_p95_compare_2x2.pdf',
    'save_percentile_maps_png': FIG_DIR / 'figure_extreme_p95_p99_seasonal_maps_2x4.png',
    'save_percentile_maps_pdf': FIG_DIR / 'figure_extreme_p95_p99_seasonal_maps_2x4.pdf',
    'save_warming_change_png': FIG_DIR / 'figure_p4k_minus_cntl_rainfall_wind_changes_3x4.png',
    'save_warming_change_pdf': FIG_DIR / 'figure_p4k_minus_cntl_rainfall_wind_changes_3x4.pdf',
}

REGION_BOXES = {
    'PM': (99.0, 104.5, 1.0, 7.0),
    'EM': (109.0, 119.0, 0.5, 6.5),
    'WI': (99.0, 105.0, -9.0, 2.0),
    'NI': (104.5, 113.0, -5.0, 1.0),
    'SI': (104.0, 115.0, -10.0, -6.0),
    'EI': (113.0, 124.0, -5.0, 2.0),
    'NP': (120.0, 126.5, 12.0, 21.0),
    'SP': (119.0, 126.5, 5.0, 13.0),
}

In [ ]:
def set_agu_style(plot=PLOT):
    mpl.rcParams.update({
        'font.family': plot['font_family'],
        'font.size': plot['base_fontsize'],
        'axes.titlesize': plot['title_fontsize'],
        'axes.labelsize': plot['base_fontsize'],
        'xtick.labelsize': plot['base_fontsize'],
        'ytick.labelsize': plot['base_fontsize'],
        'legend.fontsize': plot['base_fontsize'],
        'figure.titlesize': plot['title_fontsize'] + 1,
        'axes.linewidth': 0.6,
        'xtick.major.width': 0.6,
        'ytick.major.width': 0.6,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'savefig.dpi': plot['save_dpi'],
        'savefig.bbox': 'tight',
    })


def get_colormap(name, n, low=0.08, high=0.96, pale_first=True):
    if name.startswith('cmocean_'):
        try:
            import cmocean
            base = getattr(cmocean.cm, name.replace('cmocean_', ''))
        except (ImportError, AttributeError):
            base = plt.get_cmap('YlGnBu')
    else:
        base = plt.get_cmap(name)
    colors = base(np.linspace(low, high, n))
    if pale_first:
        colors[0] = np.array([0.985, 0.985, 0.94, 1.0])
    return ListedColormap(colors)


def precipitation_cmap(levels, cmap_name='cmocean_rain'):
    return get_colormap(cmap_name, len(levels) + 1, pale_first=True)


def comparison_cmap(levels, cmap_name='cmocean_matter'):
    return get_colormap(cmap_name, len(levels) + 1, low=0.12, high=0.96, pale_first=False)


def change_cmap(levels, cmap_name='cmocean_balance'):
    return get_colormap(cmap_name, len(levels) + 1, low=0.06, high=0.94, pale_first=False)


def _ticks_for_extent(vmin, vmax, step):
    first = np.ceil(vmin / step) * step
    last = np.floor(vmax / step) * step
    if last < first:
        return [vmin, vmax]
    return np.arange(first, last + 0.5 * step, step)


def _coord_values(da, lon_name='wind_lon', lat_name='wind_lat'):
    if lon_name in da.coords and lat_name in da.coords:
        return da[lon_name], da[lat_name]
    return da['lon'], da['lat']


def _turn_off_cartopy_ticks(ax, labelsize=15, spine_lw=1.5):
    ax.tick_params(labelsize=labelsize, direction='out', top=False, right=False)

    from matplotlib.lines import Line2D

    try:
        ax.spines['geo'].set_visible(False)
    except KeyError:
        ax.outline_patch.set_visible(False)

    plt.draw()
    ax.xaxis.set_tick_params(top=False, which='both')
    ax.yaxis.set_tick_params(right=False, which='both')
    ax.add_artist(Line2D([0, 0], [0, 1], transform=ax.transAxes,
                         color='black', linewidth=spine_lw, clip_on=False))
    ax.add_artist(Line2D([0, 1], [0, 0], transform=ax.transAxes,
                         color='black', linewidth=spine_lw, clip_on=False))


def format_geo_axis(ax, extent=DOMAIN, plot=PLOT, left_labels=True, bottom_labels=True):
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.coastlines('50m', linewidth=plot['coastline_lw'])
    ax.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=plot['border_lw'])
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(), draw_labels=True,
        linewidth=plot['grid_lw'], color='0.55', alpha=0.65, linestyle='--'
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = left_labels
    gl.bottom_labels = bottom_labels
    gl.xlocator = plt.FixedLocator(_ticks_for_extent(extent[0], extent[1], plot['lon_tick_step']))
    gl.ylocator = plt.FixedLocator(_ticks_for_extent(extent[2], extent[3], plot['lat_tick_step']))
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()
    gl.xlabel_style = {'size': plot['base_fontsize']}
    gl.ylabel_style = {'size': plot['base_fontsize']}
    _turn_off_cartopy_ticks(
        ax,
        labelsize=plot['base_fontsize'],
        spine_lw=plot.get('map_spine_lw', 1.5),
    )


def add_panel_label(ax, label, plot=PLOT):
    ax.text(
        0.015, 0.985, label, transform=ax.transAxes,
        ha='left', va='top', fontsize=plot['panel_label_fontsize'], fontweight='bold',
        bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.72, 'pad': 1.0},
        zorder=20,
    )


def add_region_boxes(ax, boxes=REGION_BOXES, plot=PLOT):
    for label, (lon0, lon1, lat0, lat1) in boxes.items():
        ax.plot(
            [lon0, lon1, lon1, lon0, lon0],
            [lat0, lat0, lat1, lat1, lat0],
            color='black', linewidth=plot['box_lw'], transform=ccrs.PlateCarree(), zorder=8,
        )
        ax.text(
            lon0 + 0.45, lat1 - 1.0, label,
            transform=ccrs.PlateCarree(), fontsize=plot['panel_label_fontsize'] - 1,
            fontweight='bold', ha='left', va='top', zorder=9,
        )


def add_wind_quiver(ax, u, v, plot=PLOT):
    step = plot['wind_step']
    wind_lon, wind_lat = _coord_values(u)
    q = ax.quiver(
        wind_lon[::step], wind_lat[::step],
        u.values[::step, ::step], v.values[::step, ::step],
        transform=ccrs.PlateCarree(), color='black', pivot='middle',
        width=plot['wind_width'], headwidth=3.2, headlength=4.0,
        headaxislength=3.5, scale=plot['wind_scale'], zorder=7,
    )
    simple_quiver_legend(
        ax, q,
        reference_value=plot['wind_key'], unit='',
        legend_location=plot['wind_legend_location'],
        box_width=0.12, box_height=0.10, text_offset=0.020,
        font_size=plot['base_fontsize'], label_separation=0.04,
        box_facecolor='white', box_edgecolor='none', box_linewidth=0.0,
        zorder=12,
    )
    return q


def draw_rain_panel(
    ax, rain, levels, cmap, title, label,
    u=None, v=None, plot=PLOT,
    left_labels=True, bottom_labels=True,
):
    norm = BoundaryNorm(levels, cmap.N, extend='both')
    cf = ax.contourf(
        rain['lon'], rain['lat'], rain,
        levels=levels, cmap=cmap, norm=norm, extend='both',
        transform=ccrs.PlateCarree(), antialiased=False,
    )
    format_geo_axis(ax, plot=plot, left_labels=left_labels, bottom_labels=bottom_labels)
    ax.set_title(title, pad=3)
    add_panel_label(ax, label, plot=plot)
    if u is not None and v is not None:
        add_wind_quiver(ax, u, v, plot=plot)
    return cf


def draw_change_panel(
    ax, field, levels, cmap, title, label,
    u=None, v=None, plot=PLOT,
    left_labels=True, bottom_labels=True,
):
    norm = BoundaryNorm(levels, cmap.N, extend='both')
    cf = ax.contourf(
        field['lon'], field['lat'], field,
        levels=levels, cmap=cmap, norm=norm, extend='both',
        transform=ccrs.PlateCarree(), antialiased=False,
    )
    format_geo_axis(ax, plot=plot, left_labels=left_labels, bottom_labels=bottom_labels)
    ax.set_title(title, pad=3)
    add_panel_label(ax, label, plot=plot)
    if u is not None and v is not None:
        add_wind_quiver(ax, u, v, plot=plot)
    return cf


def percentile_field(name, season):
    return ds[name].sel(season=season)

In [ ]:
set_agu_style(PLOT)

mean_cmap = precipitation_cmap(MEAN_LEVELS, PLOT['cmap_name'])
p95_cmap = precipitation_cmap(P95_LEVELS, PLOT['cmap_name'])

fig = plt.figure(figsize=(PLOT['figure_width_in'], PLOT['figure_height_in']))
gs = fig.add_gridspec(
    nrows=5, ncols=2,
    height_ratios=[1.0, 1.0, 0.07, 1.0, 0.07],
    hspace=0.34, wspace=0.12,
)

axes = [
    fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree()),
    fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree()),
    fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree()),
    fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree()),
    fig.add_subplot(gs[3, 0], projection=ccrs.PlateCarree()),
    fig.add_subplot(gs[3, 1], projection=ccrs.PlateCarree()),
]

mean_cf = None
for idx, season in enumerate(SEASONS):
    mean_cf = draw_rain_panel(
        axes[idx], ds['rain_mean'].sel(season=season), MEAN_LEVELS, mean_cmap,
        f'{season} mean', PANEL_LABELS[idx],
        u=ds['u850'].sel(season=season), v=ds['v850'].sel(season=season), plot=PLOT,
    )

p95_cf = draw_rain_panel(
    axes[4], percentile_field('rain_p95', 'DJF'), P95_LEVELS, p95_cmap,
    'DJF 95th percentile', PANEL_LABELS[4], plot=PLOT,
)
add_region_boxes(axes[4], plot=PLOT)

draw_rain_panel(
    axes[5], percentile_field('rain_p95', 'JJA'), P95_LEVELS, p95_cmap,
    'JJA 95th percentile', PANEL_LABELS[5], plot=PLOT,
)

cax1 = fig.add_subplot(gs[2, :])
cbar1 = fig.colorbar(mean_cf, cax=cax1, orientation='horizontal', ticks=MEAN_LEVELS)
cbar1.ax.tick_params(length=2, width=0.6)
cbar1.set_label('precipitation (mm day$^{-1}$)')

cax2 = fig.add_subplot(gs[4, :])
cbar2 = fig.colorbar(p95_cf, cax=cax2, orientation='horizontal', ticks=P95_LEVELS)
cbar2.ax.tick_params(length=2, width=0.6)
cbar2.set_label('precipitation (mm day$^{-1}$)')

period = '1980-01-01 to 1993-12-31'
fig.suptitle(f'Seasonal rainfall and 850 hPa winds, {period}', y=0.982)
fig.subplots_adjust(top=0.945, bottom=0.055, left=0.07, right=0.98)

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(PLOT['save_main_png'], dpi=PLOT['save_dpi'])
fig.savefig(PLOT['save_main_pdf'])
plt.show()
print(f"Saved: {PLOT['save_main_png']}")
print(f"Saved: {PLOT['save_main_pdf']}")

## Seasonal P95 and P99 Spatial Maps

This 2x4 figure shows the absolute 95th and 99th percentile rainfall maps for all four seasons. Each percentile row uses its own colorbar so the P99 range can extend beyond the P95 range.

In [ ]:
percentile_specs = [
    ('rain_p95', '95th percentile', P95_LEVELS, precipitation_cmap(P95_LEVELS, PLOT['cmap_name'])),
    ('rain_p99', '99th percentile', P99_LEVELS, precipitation_cmap(P99_LEVELS, PLOT['cmap_name'])),
]

fig_pct = plt.figure(figsize=(PLOT['percentile_maps_width_in'], PLOT['percentile_maps_height_in']))
gs_pct = fig_pct.add_gridspec(
    nrows=5, ncols=4,
    height_ratios=[1.0, 0.08, 0.20, 1.0, 0.08],
    hspace=0.18, wspace=0.08,
)

label_idx = 0
for pct_idx, (field_name, pct_title, levels, cmap) in enumerate(percentile_specs):
    map_row = 0 if pct_idx == 0 else 3
    row_cf = None
    for season_idx, season in enumerate(SEASONS):
        ax = fig_pct.add_subplot(gs_pct[map_row, season_idx], projection=ccrs.PlateCarree())
        row_cf = draw_rain_panel(
            ax,
            percentile_field(field_name, season),
            levels,
            cmap,
            f'{season} {pct_title}',
            PERCENTILE_MAP_LABELS[label_idx],
            plot=PLOT,
            left_labels=(season_idx == 0),
            bottom_labels=True,
        )
        label_idx += 1

    cax = fig_pct.add_subplot(gs_pct[map_row + 1, :])
    cbar = fig_pct.colorbar(row_cf, cax=cax, orientation='horizontal', ticks=levels)
    cbar.ax.tick_params(length=2, width=0.6)
    cbar.set_label(f'{pct_title} precipitation (mm day$^{{-1}}$)')

fig_pct.suptitle('Seasonal 95th and 99th percentile rainfall', y=0.985)
fig_pct.subplots_adjust(top=0.91, bottom=0.08, left=0.055, right=0.985)
fig_pct.savefig(PLOT['save_percentile_maps_png'], dpi=PLOT['save_dpi'])
fig_pct.savefig(PLOT['save_percentile_maps_pdf'])
plt.show()
print(f"Saved: {PLOT['save_percentile_maps_png']}")
print(f"Saved: {PLOT['save_percentile_maps_pdf']}")


## P4K Minus CNTL Changes

This 3x4 figure shows P4K-CNTL changes for seasonal mean rainfall, P95, and P99. The first row overlays 850 hPa wind vector changes.

In [ ]:
mean_change_cmap = change_cmap(MEAN_CHANGE_LEVELS, PLOT['change_cmap_name'])
extreme_change_cmap = change_cmap(EXTREME_CHANGE_LEVELS, PLOT['change_cmap_name'])
change_plot = dict(PLOT)
change_plot['wind_key'] = PLOT['change_wind_key']
change_plot['wind_scale'] = PLOT['change_wind_scale']

change_specs = [
    ('rain_mean', 'mean rainfall change', MEAN_CHANGE_LEVELS, mean_change_cmap, True),
    ('rain_p95', 'P95 rainfall change', EXTREME_CHANGE_LEVELS, extreme_change_cmap, False),
    ('rain_p99', 'P99 rainfall change', EXTREME_CHANGE_LEVELS, extreme_change_cmap, False),
]

fig_chg = plt.figure(figsize=(PLOT['warming_change_width_in'], PLOT['warming_change_height_in']))
gs_chg = fig_chg.add_gridspec(
    nrows=8, ncols=4,
    height_ratios=[1.0, 0.08, 0.24, 1.0, 0.08, 0.24, 1.0, 0.08],
    hspace=0.16, wspace=0.08,
)

label_idx = 0
for row_idx, (field_name, row_title, levels, cmap, with_wind) in enumerate(change_specs):
    map_row = row_idx * 3
    row_cf = None
    for season_idx, season in enumerate(SEASONS):
        ax = fig_chg.add_subplot(gs_chg[map_row, season_idx], projection=ccrs.PlateCarree())
        u_change = ds_diff['u850'].sel(season=season) if with_wind else None
        v_change = ds_diff['v850'].sel(season=season) if with_wind else None
        row_cf = draw_change_panel(
            ax,
            ds_diff[field_name].sel(season=season),
            levels,
            cmap,
            f'{season} {row_title}',
            WARMING_CHANGE_LABELS[label_idx],
            u=u_change,
            v=v_change,
            plot=change_plot,
            left_labels=(season_idx == 0),
            bottom_labels=True,
        )
        label_idx += 1

    cax = fig_chg.add_subplot(gs_chg[map_row + 1, :])
    cbar = fig_chg.colorbar(row_cf, cax=cax, orientation='horizontal', ticks=levels)
    cbar.ax.tick_params(length=2, width=0.6)
    cbar.set_label(f'P4K - CNTL {row_title} (mm day$^{{-1}}$)')

fig_chg.suptitle('P4K minus CNTL seasonal rainfall and 850 hPa wind changes', y=0.985)
fig_chg.subplots_adjust(top=0.93, bottom=0.07, left=0.055, right=0.985)
fig_chg.savefig(PLOT['save_warming_change_png'], dpi=PLOT['save_dpi'])
fig_chg.savefig(PLOT['save_warming_change_pdf'])
plt.show()
print(f"Saved: {PLOT['save_warming_change_png']}")
print(f"Saved: {PLOT['save_warming_change_pdf']}")


## 99th versus 95th percentile comparison

The default 2x2 plot shows `P99/P95` for DJF, MAM, JJA, and SON. Change `EXTREME_COMPARE_MODE` to `'difference'` above to plot `P99-P95` in mm day-1.

In [ ]:
if EXTREME_COMPARE_MODE == 'ratio':
    compare_data = ds['rain_p99'] / ds['rain_p95']
    compare_levels = RATIO_LEVELS
    compare_label = 'P99 / P95'
    compare_title = '99th-to-95th percentile rainfall ratio'
elif EXTREME_COMPARE_MODE == 'difference':
    compare_data = ds['rain_p99'] - ds['rain_p95']
    compare_levels = DIFF_LEVELS
    compare_label = 'P99 - P95 (mm day$^{-1}$)'
    compare_title = '99th minus 95th percentile rainfall'
else:
    raise ValueError("EXTREME_COMPARE_MODE must be 'ratio' or 'difference'")

compare_cmap = comparison_cmap(compare_levels, PLOT['compare_cmap_name'])
compare_norm = BoundaryNorm(compare_levels, compare_cmap.N, extend='both')

fig2 = plt.figure(figsize=(PLOT['compare_width_in'], PLOT['compare_height_in']))
gs2 = fig2.add_gridspec(nrows=3, ncols=2, height_ratios=[1.0, 1.0, 0.075], hspace=0.28, wspace=0.12)
axes2 = [
    fig2.add_subplot(gs2[0, 0], projection=ccrs.PlateCarree()),
    fig2.add_subplot(gs2[0, 1], projection=ccrs.PlateCarree()),
    fig2.add_subplot(gs2[1, 0], projection=ccrs.PlateCarree()),
    fig2.add_subplot(gs2[1, 1], projection=ccrs.PlateCarree()),
]

compare_cf = None
for idx, season in enumerate(SEASONS):
    field = compare_data.sel(season=season)
    compare_cf = axes2[idx].contourf(
        field['lon'], field['lat'], field,
        levels=compare_levels, cmap=compare_cmap, norm=compare_norm,
        extend='both', transform=ccrs.PlateCarree(), antialiased=False,
    )
    format_geo_axis(axes2[idx], plot=PLOT)
    axes2[idx].set_title(season, pad=3)
    add_panel_label(axes2[idx], COMPARE_LABELS[idx], plot=PLOT)

cax = fig2.add_subplot(gs2[2, :])
cbar = fig2.colorbar(compare_cf, cax=cax, orientation='horizontal', ticks=compare_levels)
cbar.ax.tick_params(length=2, width=0.6)
cbar.set_label(compare_label)

fig2.suptitle(compare_title, y=0.985)
fig2.subplots_adjust(top=0.935, bottom=0.075, left=0.07, right=0.98)
fig2.savefig(PLOT['save_compare_png'], dpi=PLOT['save_dpi'])
fig2.savefig(PLOT['save_compare_pdf'])
plt.show()
print(f"Saved: {PLOT['save_compare_png']}")
print(f"Saved: {PLOT['save_compare_pdf']}")